# Vector Databases and Graph Mechanics

### Objective
Understand the mechanics of modern approximate nearest neighbotr (ANN) search by building a simplified version of a \
Hierarchical Navigable Small World (HNSW) graph index from scratch. Instead of using a flat exhaustive search, which \
scales poorly at $O(N)$ as databases grow, you will implement a multi-layer graph skip-list structure\
that guides search queries through dense vector spaces in log0time $O(log N)$.

### Core Architecture (Constraints)
- No Vector Search Packages: Do not use faiss, chromadb, scipy.spatial, or scikit-learn. Implement the distance metrics,   
mutli-layer graph routing, and entry point trcking using standard Python data structures and pure numpy.

- Multi-layer Skip-Graph Mechanics: Implement a 2-layer graph layout:
    - Layer 1 (Top Layer / Express): Contains a sparse subset of points to bridge long spatial distances quickly.
    
    - Layer 0 (Bottom Layer / Dense): Contains every vector node in the dataset for granular, localised resolution.

### Engineering Requirements
1. Distance Metric Engine\
Implement standard Cosine Similarity or Euclidean Distance $(L_2)$ using purew vectorized NumPy arrays. This will serve\
as your routing metric when scoring candidate vectors.

2. Multi-Layer Insertion Loop\
Write a sequetial graph construction algorithm:
- Each node is assigned a maximum layer height. For this 2-layer implementation, force your coordinate seed data to  
balance evenly across both tiers.

- When adding a new node, start at the top layer (Layer 1), locate the nearest existing neighbor, and use that match as  
the entry point backpointer to begin a more localized nearest-neighbor search on the lower layer (Layer 0).

3. Greedy Graph Routing Search Tactic\
Write a graph traversal query function:
- Start at the designateds entry point on Layer 1.

- Evaluate the distance from the query vector to all adjacent neighbors connected to the current node.

- Step to the neighbor that is closest to the query string. Repeat until a local minimum is hit (i.e., no connected  
neighbor is closer than the current node).

- Drop down to Layer 0 at that identical node index, and continue the greedy neighborhood sweep until a final closest  
match is trapped.

### Sample Seed Validation Data
Use these coordinte vector arrays to build, insert, and search your graph structures:

In [8]:
import numpy as np
# Raw embedding simulation coordinates (Dimensionality d = 4)
node_vectors = {
    0: np.array([0.1, 0.9, 0.0, 0.2]), # representing "text parsing",
    1: np.array([0.1, 0.8, 0.1, 0.1]), # representing "string clean",
    2: np.array([0.8, 0.2, 0.9, 0.0]), # representing "vector database",
    3: np.array([0.7, 0.1, 0.8, 0.1]) # representing "graph index"
}

# Graph Edges Layout Defintition Structure:
# Layer 1 (Express Run): Connect Node 0 <-> Node 2
# Layer 0 (Dense Base): Connect Node 0 <-> Node 1, Node 2 <-> Node 3, Node 1 <-> Node 3

# Query Vector to search for:
query = np.array([0.75, 0.15, 0.85, 0.05]) # targeted to map close to "vector database" / "graph index"

### Expected Output Layout
```
HNSW GRAPH APPROXIMATE NEAREST NEIGHBOR (v1)

INDEX CONFIGURATION
Total Indexed Nodes: 4
Layer 1 (Express) Nodes: [0, 2]
Layer 0 (Dense Base) Nodes: [0, 1, 2, 3]

GREEDY ROUTING TRACE
-> Starting at Layer 1 Entry Point: Node 0
-> Evaluating Layer 1 Adjacencies... Node 2 is closer. Moving to Node 2.
-> Dropping to Layer 0 at Node 2.
-> Evaluating Layer 0 Adjacencies... Checking Node 3.

FINAL NEAREST NEIGHBOR RESULT
Query Vector Target Match: Node 2
Cosine Simliarity Score: ...
```

### Imports
Import already completed above (import numpy as np)

### Vector Dataset
Also Supplied in the instructions

### Distance Metric Engine

In [9]:
def cosine_similarity(vector_a, vector_b):
    """Compute cosine similarity using pure NumPy."""
    numerator = np.dot(vector_a, vector_b)
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)

    if denominator ==  0:
        return 0.0
    
    return numerator / denominator

### Two-Layer Graph Constuction

In [10]:
def build_hnsw_graph():
    """Construct a simplified two-layer HNSW graph."""
    graph = {
        1: {0: [2], 2: [0]}, 0: {0: [1], 1: [0, 3], 2: [3], 3: [2, 1]}
    }
    return graph

### Greedy Routing Engine

In [11]:
def greedy_search(graph, vectors, query, entry_node=0):
    """Greedy HNSW traversal across two graph layers."""
    trace = []
    current = entry_node
    
    trace.append(f"Starting at Layer 1 Entry Point: Node {current}")

    while True:
        best_node = current
        best_score = cosine_similarity(query, vectors[current])

        for neighbor in graph[1][current]:
            score = cosine_similarity(query, vectors[neighbor])

            if score > best_score:
                best_node = neighbor
                best_score = score

        if best_node == current:
            break

        trace.append(
            f"Evaluating Layer 1 Adjacencies... "
            f"Node {best_node} is closer."
            f"Moving to Node {best_node}."
        )

        current = best_node

    trace.append(f"Dropping to Layer 0 at Node {current}.")
    
    visited = set()

    while True:
        visited.add(current)
        best_node = current
        best_score = cosine_similarity(query, vectors[current])

        for neighbor in graph[0][current]:
            if neighbor in visited:
                continue

            score = cosine_similarity(query, vectors[neighbor])
            trace.append(f"Evaluating Layer 0 Adjacencies... Checking Node {neighbor}.")

            if score > best_score:
                best_node = neighbor
                best_score = score

        if best_node == current:
            break

        current = best_node

    return current, best_score, trace

### Evaluation Harness

In [12]:
def evaluate_hnsw():
    print("HNSW GRAPH APPROXIMATE NEAREST NEIGHBOR (v1)\n")

    graph = build_hnsw_graph()

    print("INDEX CONFIGURATION")
    print(f"Total Indexed Nodes: {len(node_vectors)}")
    print(f"Layer 1 (Express) Nodes: {list(graph[1].keys())}")
    print(f"Layer 0 (Dense Base) Nodes: {list(graph[0].keys())}\n")

    node, score, trace = greedy_search(graph,node_vectors,query)

    print("GREEDY ROUTING TRACE")

    for step in trace:
        print(f"-> {step}")

    print("\nFINAL NEAREST NEIGHBOR RESULT")

    print(f"Query Vector Target Match: Node {node}")

    print(
        f"Cosine Similarity Score: {score:.4f}")

### Execute Pipeline

In [13]:
evaluate_hnsw()

HNSW GRAPH APPROXIMATE NEAREST NEIGHBOR (v1)

INDEX CONFIGURATION
Total Indexed Nodes: 4
Layer 1 (Express) Nodes: [0, 2]
Layer 0 (Dense Base) Nodes: [0, 1, 2, 3]

GREEDY ROUTING TRACE
-> Starting at Layer 1 Entry Point: Node 0
-> Evaluating Layer 1 Adjacencies... Node 2 is closer.Moving to Node 2.
-> Dropping to Layer 0 at Node 2.
-> Evaluating Layer 0 Adjacencies... Checking Node 3.

FINAL NEAREST NEIGHBOR RESULT
Query Vector Target Match: Node 2
Cosine Similarity Score: 0.9985
